In [1]:
# ============================================================
# CELDA 1 — Imports + rutas + helpers de nombres
# ============================================================
import os
from pathlib import Path
import numpy as np
import nibabel as nib

# -------------------------
# RUTAS
# -------------------------
SRC_DIR = Path("/users/lvelez/PI_Velez/data/Couinaud/CouinaudTr")
OUT_DIR = SRC_DIR / "32slices"   # se crea si no existe

OUT_DIR.mkdir(parents=True, exist_ok=True)
print("SRC_DIR:", SRC_DIR)
print("OUT_DIR:", OUT_DIR)

# -------------------------
# Helpers de nombres
# -------------------------
def base_name(fname: str) -> str:
    """
    Quita extensión .nii.gz o .nii y el sufijo '_ventaneo' (si está),
    y devuelve el nombre base.
    """
    name = fname
    if name.endswith(".nii.gz"):
        name = name[:-7]
    elif name.endswith(".nii"):
        name = name[:-4]
    else:
        name = os.path.splitext(name)[0]
    if name.endswith("_ventaneo"):
        name = name[:-9]
    return name

def build_couinaud_matches(split_dir: str):
    """
    Matchea:
      - CT ventaneadas: *_ventaneo.nii.gz
      - Máscaras Couinaud: *.nii (sin gz)
    por base_name y chequea que shapes coincidan.
    """
    split_dir = Path(split_dir)
    all_files = os.listdir(split_dir)

    ct_files = [f for f in all_files if f.endswith(".nii.gz") and ("ventaneo" in f)]
    ms_files = [f for f in all_files if f.endswith(".nii") and not f.endswith(".nii.gz")]

    ct_by_base = {base_name(f): f for f in ct_files}
    ms_by_base = {base_name(f): f for f in ms_files}

    common = sorted(set(ct_by_base.keys()) & set(ms_by_base.keys()))

    matched = []
    for b in common:
        img_path  = split_dir / ct_by_base[b]
        mask_path = split_dir / ms_by_base[b]

        img_nii  = nib.load(str(img_path))
        mask_nii = nib.load(str(mask_path))

        if img_nii.shape != mask_nii.shape:
            print(f"[OMIT] Shape mismatch {b}: img={img_nii.shape} mask={mask_nii.shape}")
            continue

        matched.append((b, img_path, mask_path))

    print(f"[build_couinaud_matches] {split_dir} -> matcheados: {len(matched)}")
    return matched

SRC_DIR: /users/lvelez/PI_Velez/data/Couinaud/CouinaudTr
OUT_DIR: /users/lvelez/PI_Velez/data/Couinaud/CouinaudTr/32slices


In [2]:
# ============================================================
# CELDA 2 — Estrategia de ventanas (Z TOTAL) + padding extractor
# ============================================================

def choose_window_starts_by_Z(Z: int, win: int = 32):
    """
    Estrategia basada en Z TOTAL para minimizar repetición:
      - Z <= 32: 1 ventana (padding si Z < 32)
      - 33..48: 1 ventana (centrada)
      - 49..80: 2 ventanas (inicio y final)
      - >=81  : 3 ventanas (inicio, medio, final)
    Devuelve lista de z_start.
    """
    if Z <= win:
        return [0]

    max_start = Z - win

    if Z <= 48:
        s = max_start // 2
        return [int(s)]

    if Z <= 80:
        return [0, int(max_start)]

    # Z >= 81 -> 3 ventanas espaciadas
    s1 = 0
    s3 = int(max_start)
    s2 = int(round(max_start / 2))
    return sorted(set([s1, s2, s3]))


def extract_window_with_padding(vol_h_w_z: np.ndarray, start: int, win: int = 32, pad_value=0):
    """
    Extrae [H,W,win] desde start.
    Si Z < win, devuelve padding al final.
    Si start+win excede Z, también pad al final (por seguridad).
    """
    H, W, Z = vol_h_w_z.shape
    end = start + win

    # Caso normal
    if end <= Z:
        return vol_h_w_z[:, :, start:end]

    # Caso padding (Z < win o start mal)
    out = np.full((H, W, win), pad_value, dtype=vol_h_w_z.dtype)
    valid = max(0, Z - start)
    if valid > 0:
        out[:, :, :valid] = vol_h_w_z[:, :, start:Z]
    return out

In [3]:
'''
# ============================================================
# CELDA 3 — Generar y guardar ventanas 32-slices (CT + máscara)
#          - Guarda TODO como .nii (sin gz)
#          - Si hay 1 ventana: nombre original
#          - Si hay >1: sufijos _1, _2, _3...
# ============================================================

WIN = 32

matched = build_couinaud_matches(str(SRC_DIR))

n_patients = 0
n_windows_total = 0

for (base, ct_path, mask_path) in matched:
    n_patients += 1

    # Cargar NIfTI (H,W,Z)
    ct_nii = nib.load(str(ct_path))
    ms_nii = nib.load(str(mask_path))

    ct = ct_nii.get_fdata().astype(np.float32)           # [H,W,Z] en [0,1]
    ms = np.round(ms_nii.get_fdata()).astype(np.int16)   # [H,W,Z] labels 0..8

    H, W, Z = ct.shape

    # Elegir ventanas por Z TOTAL (sin rango útil)
    starts = choose_window_starts_by_Z(Z, win=WIN)
    nW = len(starts)
    n_windows_total += nW

    for k, s in enumerate(starts, start=1):
        ct_win = extract_window_with_padding(ct, start=s, win=WIN, pad_value=0.0).astype(np.float32)
        ms_win = extract_window_with_padding(ms, start=s, win=WIN, pad_value=0).astype(np.int16)

        # Naming: original o original_1/_2/_3...
        if nW == 1:
            out_ct = OUT_DIR / f"{base}_ventaneo.nii"
            out_ms = OUT_DIR / f"{base}.nii"
        else:
            out_ct = OUT_DIR / f"{base}_ventaneo_{k}.nii"
            out_ms = OUT_DIR / f"{base}_{k}.nii"

        # Guardar (usar affine; mejor no arrastrar header para evitar warnings por dims)
        nib.save(nib.Nifti1Image(ct_win, affine=ct_nii.affine), str(out_ct))
        nib.save(nib.Nifti1Image(ms_win, affine=ms_nii.affine), str(out_ms))

    if n_patients % 10 == 0:
        print(f"[progreso] pacientes={n_patients}/{len(matched)} | ventanas_totales={n_windows_total}")

print("\n=== LISTO ===")
print("Pacientes procesados:", n_patients)
print("Ventanas generadas (total):", n_windows_total)
print("Salida en:", OUT_DIR)
'''

'\n# ============================================================\n# CELDA 3 — Generar y guardar ventanas 32-slices (CT + máscara)\n#          - Guarda TODO como .nii (sin gz)\n#          - Si hay 1 ventana: nombre original\n#          - Si hay >1: sufijos _1, _2, _3...\n# ============================================================\n\nWIN = 32\n\nmatched = build_couinaud_matches(str(SRC_DIR))\n\nn_patients = 0\nn_windows_total = 0\n\nfor (base, ct_path, mask_path) in matched:\n    n_patients += 1\n\n    # Cargar NIfTI (H,W,Z)\n    ct_nii = nib.load(str(ct_path))\n    ms_nii = nib.load(str(mask_path))\n\n    ct = ct_nii.get_fdata().astype(np.float32)           # [H,W,Z] en [0,1]\n    ms = np.round(ms_nii.get_fdata()).astype(np.int16)   # [H,W,Z] labels 0..8\n\n    H, W, Z = ct.shape\n\n    # Elegir ventanas por Z TOTAL (sin rango útil)\n    starts = choose_window_starts_by_Z(Z, win=WIN)\n    nW = len(starts)\n    n_windows_total += nW\n\n    for k, s in enumerate(starts, start=1):\n   

In [4]:
# ============================================================
# CELDA — ANÁLISIS GLOBAL DEL DATASET 32slices (TODOS)
#   - Cuenta CT y máscaras
#   - Verifica pares CT<->mask por key
#   - Chequea shapes (deberían ser 512x512x32)
#   - Resumen de ventanas por paciente (1/2/3)
#   - Cuántas ventanas tienen 0 slices con hígado (todo fondo)
#   - Top N ventanas con menos/más hígado (por #slices con fg)
# ============================================================

from collections import defaultdict, Counter

OUT_DIR_FINAL = SRC_DIR / "32slices"  # <- carpeta final donde generaste todo

print("Analizando carpeta:", OUT_DIR_FINAL)
assert OUT_DIR_FINAL.exists(), f"No existe: {OUT_DIR_FINAL}"

# -------------------------
# Listado de archivos
# -------------------------
ct_files = sorted(OUT_DIR_FINAL.glob("*_ventaneo*.nii"))
ms_files = sorted([p for p in OUT_DIR_FINAL.glob("*.nii") if "_ventaneo" not in p.name])

print("CT ventanas:", len(ct_files))
print("Masks      :", len(ms_files))

# -------------------------
# Claves de emparejado
#   CT:  hepaticvessel_001_ventaneo_2 -> key: hepaticvessel_001_2
#   CT:  hepaticvessel_001_ventaneo   -> key: hepaticvessel_001
#   MS:  hepaticvessel_001_2          -> key: hepaticvessel_001_2
#   MS:  hepaticvessel_001            -> key: hepaticvessel_001
# -------------------------
def strip_nii(n: str) -> str:
    return n[:-4] if n.endswith(".nii") else n

def key_from_ct(p: Path) -> str:
    n = strip_nii(p.name)
    # sacamos "_ventaneo" (una sola vez)
    n = n.replace("_ventaneo", "")
    # si quedó doble guión bajo por reemplazo raro, limpiamos (por las dudas)
    n = n.replace("__", "_")
    # si termina en "_", lo quitamos
    if n.endswith("_"):
        n = n[:-1]
    return n

def key_from_ms(p: Path) -> str:
    return strip_nii(p.name)

ct_key_to_path = {key_from_ct(p): p for p in ct_files}
ms_key_to_path = {key_from_ms(p): p for p in ms_files}

ct_keys = set(ct_key_to_path.keys())
ms_keys = set(ms_key_to_path.keys())

pairs = sorted(ct_keys & ms_keys)
only_ct = sorted(ct_keys - ms_keys)
only_ms = sorted(ms_keys - ct_keys)

print("\n=== PARES CT/MASK ===")
print("Pares OK       :", len(pairs))
print("CT sin máscara :", len(only_ct))
print("Máscara sin CT :", len(only_ms))

if len(only_ct) > 0:
    print("Ejemplo CT sin máscara:", only_ct[:5])
if len(only_ms) > 0:
    print("Ejemplo máscara sin CT:", only_ms[:5])

# -------------------------
# Agrupar por paciente base y contar ventanas
#   base: hepaticvessel_001_2 -> patient_base: hepaticvessel_001
#   base: hepaticvessel_001   -> patient_base: hepaticvessel_001
# -------------------------
# ============================================================
# FIX: agrupar correctamente por paciente
#   - patient_base = "hepaticvessel_XXX" (primeros 2 tokens)
#   - window_key   = "hepaticvessel_XXX" o "hepaticvessel_XXX_N"
# ============================================================

def patient_base_from_key(k: str) -> str:
    """
    k ejemplos:
      - hepaticvessel_031
      - hepaticvessel_031_2
    Devuelve SIEMPRE: hepaticvessel_031
    """
    parts = k.split("_")
    if len(parts) >= 2:
        return parts[0] + "_" + parts[1]
    return k  # fallback

def window_index_from_key(k: str):
    """
    Devuelve None si no hay sufijo de ventana, o int si existe.
    """
    parts = k.split("_")
    if len(parts) >= 3 and parts[-1].isdigit():
        return int(parts[-1])
    return None

# -------------------------
# Recorrer pares y medir
# -------------------------
shape_counter = Counter()
windows_per_patient = defaultdict(int)

all_window_stats = []  # lista de dicts por ventana
n_bad_shape = 0
n_all_background = 0

for k in pairs:
    ct_path = ct_key_to_path[k]
    ms_path = ms_key_to_path[k]

    ct_nii = nib.load(str(ct_path))
    ms_nii = nib.load(str(ms_path))

    ct_shape = ct_nii.shape
    ms_shape = ms_nii.shape

    shape_counter[(ct_shape, ms_shape)] += 1

    # shapes esperadas
    if ct_shape != (512, 512, 32) or ms_shape != (512, 512, 32):
        n_bad_shape += 1

    # contar slices con hígado (usamos máscara)
    ms = np.round(ms_nii.get_fdata()).astype(np.int16)  # [H,W,32]
    slices_fg = int(np.sum(np.any(ms > 0, axis=(0, 1))))
    if slices_fg == 0:
        n_all_background += 1

    # contar ventanas por paciente
    pbase = patient_base_from_key(k)
    windows_per_patient[pbase] += 1

    all_window_stats.append({
        "key": k,
        "patient": pbase,
        "ct_file": ct_path.name,
        "ms_file": ms_path.name,
        "ct_shape": ct_shape,
        "ms_shape": ms_shape,
        "slices_fg": slices_fg
    })

print("\n=== SHAPES ===")
print("Combinaciones (ct_shape, ms_shape) encontradas:", len(shape_counter))
print("Ventanas con shape incorrecto:", n_bad_shape)

# Mostrar las 5 combinaciones más frecuentes
for (shpair, cnt) in shape_counter.most_common(5):
    print(" ", shpair, "->", cnt)

print("\n=== FOREGROUND (hígado) ===")
print("Ventanas totales (pares):", len(pairs))
print("Ventanas todo-fondo (0 slices con hígado):", n_all_background)
if len(pairs) > 0:
    print("Porcentaje todo-fondo:", 100.0 * n_all_background / len(pairs), "%")

# -------------------------
# Resumen ventanas por paciente
# -------------------------
counts = Counter(windows_per_patient.values())
print("\n=== VENTANAS POR PACIENTE ===")
print("Pacientes (bases) detectados:", len(windows_per_patient))
for nwin in sorted(counts.keys()):
    print(f"  {nwin} ventana(s): {counts[nwin]} pacientes")

# Top pacientes con más ventanas (por si algo raro pasó)
top_pat = sorted(windows_per_patient.items(), key=lambda x: x[1], reverse=True)[:10]
print("\nTop pacientes por #ventanas (debug):")
for pb, nw in top_pat:
    print(" ", pb, "->", nw)

# -------------------------
# Top ventanas por slices_fg
# -------------------------
all_window_stats_sorted = sorted(all_window_stats, key=lambda d: d["slices_fg"])

print("\n=== TOP 10 ventanas con MENOS hígado (slices_fg) ===")
for d in all_window_stats_sorted[:10]:
    print(f"  {d['key']} | slices_fg={d['slices_fg']:2d}/32 | CT={d['ct_file']} | MS={d['ms_file']}")

Analizando carpeta: /users/lvelez/PI_Velez/data/Couinaud/CouinaudTr/32slices
CT ventanas: 260
Masks      : 260

=== PARES CT/MASK ===
Pares OK       : 260
CT sin máscara : 0
Máscara sin CT : 0

=== SHAPES ===
Combinaciones (ct_shape, ms_shape) encontradas: 1
Ventanas con shape incorrecto: 0
  ((512, 512, 32), (512, 512, 32)) -> 260

=== FOREGROUND (hígado) ===
Ventanas totales (pares): 260
Ventanas todo-fondo (0 slices con hígado): 14
Porcentaje todo-fondo: 5.384615384615385 %

=== VENTANAS POR PACIENTE ===
Pacientes (bases) detectados: 161
  1 ventana(s): 92 pacientes
  2 ventana(s): 39 pacientes
  3 ventana(s): 30 pacientes

Top pacientes por #ventanas (debug):
  hepaticvessel_031 -> 3
  hepaticvessel_033 -> 3
  hepaticvessel_138 -> 3
  hepaticvessel_141 -> 3
  hepaticvessel_171 -> 3
  hepaticvessel_173 -> 3
  hepaticvessel_175 -> 3
  hepaticvessel_207 -> 3
  hepaticvessel_213 -> 3
  hepaticvessel_219 -> 3

=== TOP 10 ventanas con MENOS hígado (slices_fg) ===
  hepaticvessel_138_1 | 